In [1]:
import os
import time
import random
import warnings

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from lxml import etree # type: ignore <- pylance milně hlásí chybu
from pathlib import Path
import time
import sys
import polars as pl
import polars.selectors as cs
import json
import pickle
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from ydata_profiling import ProfileReport

from utils import *
from schemas import *
from processing import *

pl.Config.set_tbl_cols(-1)
os.chdir(r'E:\CVUT_BAP')
SEED=42

# Načtení dat

In [2]:
df = pl.read_parquet('kod/data/prohlidky_vozidel_stk_a_sme/parquet/prohlidky', schema=prohlidky_schema)#.sample(fraction=1.0, shuffle=True, seed=SEED)
short_display(df)
display_counts(df)

(48338032, 52)


,CisloProtokolu,DatumProhlidky,DruhProhlidky,RozsahProhlidky,Prohlidka_OdpovednaOsoba,Prohlidka_Stanice_Cislo,Prohlidka_Stanice_Kraj,Prohlidka_Stanice_ORP,Prohlidka_Stanice_Obec,Prohlidka_Zahajeni,...,Adr_KodCisterny,Adr_CisloOsvedceni,Adr_ZavadyText,Adr_Poznamka,Tsk_OdpovednaOsoba,Vysledek_Odometr,Vysledek_Poznamka,Vysledek_DatumPristiProhlidky,Vysledek_NalepkaVylepena,Vysledek_Celkovy
0,CZ-520406-19-01-0001,2019-01-01,Pravidelná,None,16835,520406,Středočeský kraj,Kolín,Žabonosy,2019-01-01T15:21:49.0770000+01:00,...,None,None,None,None,None,309068,None,2021-01-01,None,1
1,CZ-420930-19-01-0001,2019-01-01,Pravidelná,None,43281,420930,Středočeský kraj,Říčany,Říčany,2019-01-01T14:35:04.4400000+01:00,...,None,None,None,None,None,169952,None,2021-01-01,None,1
2,CZ-6711-20-51-9008,2020-01-01,Pravidelná,Plný,28247,6711,Kraj Vysočina,Jihlava,Dolní Cerekev,2020-01-01T00:00:00.0000000+01:00,...,None,None,None,None,None,None,None,2024-01-01,true,1
3,CZ-6711-20-51-9009,2020-01-01,Pravidelná,Plný,28247,6711,Kraj Vysočina,Jihlava,Dolní Cerekev,2020-01-01T00:00:00.0000000+01:00,...,None,None,None,None,None,None,None,2024-01-01,true,1
4,CZ-6711-20-51-9015,2020-01-01,Pravidelná,Plný,28247,6711,Kraj Vysočina,Jihlava,Dolní Cerekev,2020-01-01T00:00:00.0000000+01:00,...,None,None,None,None,None,None,None,2024-01-01,true,1
5,CZ-6711-20-51-9010,2020-01-01,Pravidelná,Plný,28247,6711,Kraj Vysočina,Jihlava,Dolní Cerekev,2020-01-01T00:00:00.0000000+01:00,...,None,None,None,None,None,None,None,2024-01-01,true,1
6,CZ-6711-20-51-9011,2020-01-01,Pravidelná,Plný,28247,6711,Kraj Vysočina,Jihlava,Dolní Cerekev,2020-01-01T00:00:00.0000000+01:00,...,None,None,None,None,None,None,None,2024-01-01,true,1
7,CZ-6711-20-51-9012,2020-01-01,Pravidelná,Plný,28247,6711,Kraj Vysočina,Jihlava,Dolní Cerekev,2020-01-01T00:00:00.0000000+01:00,...,None,None,None,None,None,None,None,2024-01-01,true,1
8,CZ-6711-20-51-9013,2020-01-01,Pravidelná,Plný,28247,6711,Kraj Vysočina,Jihlava,Dolní Cerekev,2020-01-01T00:00:00.0000000+01:00,...,None,None,None,None,None,None,None,2024-01-01,true,1
9,CZ-6711-20-51-9014,2020-01-01,Pravidelná,Plný,28247,6711,Kraj Vysočina,Jihlava,Dolní Cerekev,2020-01-01T00:00:00.0000000+01:00,...,None,None,None,None,None,None,None,2024-01-01,true,1


,0
CisloProtokolu,48338032 / 48338032
DatumProhlidky,48338032 / 48338032
DruhProhlidky,48338032 / 48338032
RozsahProhlidky,29226013 / 48338032
Prohlidka_OdpovednaOsoba,48338032 / 48338032
Prohlidka_Stanice_Cislo,48338032 / 48338032
Prohlidka_Stanice_Kraj,48338032 / 48338032
Prohlidka_Stanice_ORP,48333234 / 48338032
Prohlidka_Stanice_Obec,48333234 / 48338032
Prohlidka_Zahajeni,48338032 / 48338032


# Výběr relevantních sloupců a řádků

In [3]:
df = df.filter(pl.col('Emise_CisloProtokolu').is_not_null())
describe(df)

(20060291, 52)


,CisloProtokolu,DatumProhlidky,DruhProhlidky,RozsahProhlidky,Prohlidka_OdpovednaOsoba,Prohlidka_Stanice_Cislo,Prohlidka_Stanice_Kraj,Prohlidka_Stanice_ORP,Prohlidka_Stanice_Obec,Prohlidka_Zahajeni,...,Adr_KodCisterny,Adr_CisloOsvedceni,Adr_ZavadyText,Adr_Poznamka,Tsk_OdpovednaOsoba,Vysledek_Odometr,Vysledek_Poznamka,Vysledek_DatumPristiProhlidky,Vysledek_NalepkaVylepena,Vysledek_Celkovy
0,CZ-520406-19-01-0001,2019-01-01,Pravidelná,None,16835,520406,Středočeský kraj,Kolín,Žabonosy,2019-01-01T15:21:49.0770000+01:00,...,None,None,None,None,None,309068,None,2021-01-01,None,1
1,CZ-420930-19-01-0001,2019-01-01,Pravidelná,None,43281,420930,Středočeský kraj,Říčany,Říčany,2019-01-01T14:35:04.4400000+01:00,...,None,None,None,None,None,169952,None,2021-01-01,None,1
2,CZ-480811-23-01-0001,2023-01-01,Pravidelná,None,36479,480811,Olomoucký kraj,Přerov,Kojetín,2023-01-01T12:21:49.8570000+01:00,...,None,None,None,None,None,241379,None,2025-01-01,None,1
3,CZ-420221-23-01-0004,2023-01-01,Pravidelná,None,44365,420221,Středočeský kraj,Beroun,Bavoryně,2023-01-01T17:25:41.6570000+01:00,...,None,None,None,None,None,65923,None,2025-01-01,None,1
4,CZ-420221-23-01-0001,2023-01-01,Pravidelná,None,44365,420221,Středočeský kraj,Beroun,Bavoryně,2023-01-01T16:05:05.4600000+01:00,...,None,None,None,None,None,264005,None,2025-01-01,None,1
5,CZ-420221-23-01-0003,2023-01-01,Pravidelná,None,44365,420221,Středočeský kraj,Beroun,Bavoryně,2023-01-01T16:56:47.6530000+01:00,...,None,None,None,None,None,146446,None,2025-01-01,None,1
6,CZ-170901-23-01-0001,2023-01-01,Pravidelná,None,89493,170901,Olomoucký kraj,Prostějov,Bedihošť,2023-01-01T19:23:52.2530000+01:00,...,None,None,None,None,None,237886,None,2025-01-01,None,1
7,CZ-170901-23-01-0002,2023-01-01,Pravidelná,None,89493,170901,Olomoucký kraj,Prostějov,Bedihošť,2023-01-01T19:40:36.1970000+01:00,...,None,None,None,None,None,210028,None,2025-01-01,None,1
8,CZ-520406-24-01-0001,2024-01-01,Pravidelná,None,1308,520406,Středočeský kraj,Kolín,Žabonosy,2024-01-01T11:30:09.4530000+01:00,...,None,None,None,None,None,237115,None,2026-01-01,None,1
9,CZ-480811-24-01-0001,2024-01-01,Pravidelná,None,36479,480811,Olomoucký kraj,Přerov,Kojetín,2024-01-01T16:21:48.9530000+01:00,...,None,None,None,None,None,263886,None,2026-01-01,None,1


,0
CisloProtokolu,20060291 / 20060291
DatumProhlidky,20060291 / 20060291
DruhProhlidky,20060291 / 20060291
RozsahProhlidky,948272 / 20060291
Prohlidka_OdpovednaOsoba,20060291 / 20060291
Prohlidka_Stanice_Cislo,20060291 / 20060291
Prohlidka_Stanice_Kraj,20060291 / 20060291
Prohlidka_Stanice_ORP,20060291 / 20060291
Prohlidka_Stanice_Obec,20060291 / 20060291
Prohlidka_Zahajeni,20060291 / 20060291


CisloProtokolu,DatumProhlidky,DruhProhlidky,RozsahProhlidky,Prohlidka_OdpovednaOsoba,Prohlidka_Stanice_Cislo,Prohlidka_Stanice_Kraj,Prohlidka_Stanice_ORP,Prohlidka_Stanice_Obec,Prohlidka_Zahajeni,Prohlidka_Ukonceni,AdministrativniOprava_CisloProtokolu,AdministrativniOprava_DatumProhlidky,Vozidlo_Vin,Vozidlo_Druh,Vozidlo_Kategorie,Vozidlo_Provedeni,Vozidlo_Znacka,Vozidlo_ObchodniOznaceni,Vozidlo_TypMotoru,Registrace_DatumPrvni,Registrace_Stat,Registrace_CisloDokladu,Emise_CisloProtokolu,Emise_DatumProhlidky,Emise_Stanice_Cislo,Emise_Zahajeni,Emise_Ukonceni,Emise_OdpovednaOsoba,Emise_ZakladniPalivo,Emise_AlternativniPalivo,Emise_EmisniSystem,Emise_VyrobceMotoru,Emise_CisloMotoru,Technicka_Zahajeni,Technicka_Ukonceni,Technicka_OdpovednaOsoba,Adr_Zahajeni,Adr_Ukonceni,Adr_OdpovednaOsoba,Adr_Platnost_Periodicka,Adr_Platnost_Meziperiodicka,Adr_KodCisterny,Adr_CisloOsvedceni,Adr_ZavadyText,Adr_Poznamka,Tsk_OdpovednaOsoba,Vysledek_Odometr,Vysledek_Poznamka,Vysledek_DatumPristiProhlidky,Vysledek_NalepkaVylepena,Vysledek_Celkovy
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""CZ-520406-19-01-0001""","""2019-01-01""","""Pravidelná""",null,"""16835""","""520406""","""Středočeský kraj""","""Kolín""","""Žabonosy""","""2019-01-01T15:21:49.0770000+01…","""2019-01-01T15:40:05.4700000+01…",null,null,"""WV1ZZZ7HZ7H079425""","""NÁKLADNÍ AUTOMOBIL""","""N1""",null,"""VW""","""TRANSPORTER""","""BPC""","""2007-01-03T00:00:00.0000000+01…","""Česká republika""","""UB654181""","""CZ-520406-19-01-0001""","""2019-01-01T15:40:05.4700000+01…","""520406""","""2019-01-01T15:22:33.2900000+01…","""2019-01-01T15:39:51.5500000+01…","""16835""","""Nafta""",null,"""Řízený s OBD""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""309068""",null,"""2021-01-01""",null,"""1"""


Přítomnost sloupců v případě nenastání souběžné technické prohlídky

In [ ]:
describe(df.filter(pl.col('Technicka_OdpovednaOsoba').is_null()))

(948272, 52)


,CisloProtokolu,DatumProhlidky,DruhProhlidky,RozsahProhlidky,Prohlidka_OdpovednaOsoba,Prohlidka_Stanice_Cislo,Prohlidka_Stanice_Kraj,Prohlidka_Stanice_ORP,Prohlidka_Stanice_Obec,Prohlidka_Zahajeni,...,Adr_KodCisterny,Adr_CisloOsvedceni,Adr_ZavadyText,Adr_Poznamka,Tsk_OdpovednaOsoba,Vysledek_Odometr,Vysledek_Poznamka,Vysledek_DatumPristiProhlidky,Vysledek_NalepkaVylepena,Vysledek_Celkovy
0,CZ-3848-19-02-0017,2019-02-01,Pravidelná,Plný,49369,3848,Moravskoslezský kraj,Kopřivnice,Kopřivnice,2019-02-01T10:12:21.8470000+01:00,...,None,None,None,None,None,223122,None,2021-02-01,true,1
1,CZ-3848-19-02-0012,2019-02-01,Pravidelná,Plný,49369,3848,Moravskoslezský kraj,Kopřivnice,Kopřivnice,2019-02-01T08:35:53.7000000+01:00,...,None,None,None,None,None,212424,None,2021-02-01,true,1
2,CZ-3848-19-02-0003,2019-02-01,Pravidelná,Plný,49369,3848,Moravskoslezský kraj,Kopřivnice,Kopřivnice,2019-02-01T07:04:49.5230000+01:00,...,None,None,None,None,None,73587,None,2021-02-01,true,1
3,CZ-3848-19-02-0010,2019-02-01,Pravidelná,Plný,49369,3848,Moravskoslezský kraj,Kopřivnice,Kopřivnice,2019-02-01T08:02:32.0730000+01:00,...,None,None,None,None,None,324927,None,2021-02-01,true,1
4,CZ-3848-19-02-0014,2019-02-01,Pravidelná,Plný,49369,3848,Moravskoslezský kraj,Kopřivnice,Kopřivnice,2019-02-01T09:04:03.5400000+01:00,...,None,None,None,None,None,163604,None,2021-02-01,true,1
5,CZ-3848-19-02-0008,2019-02-01,Pravidelná,Plný,49369,3848,Moravskoslezský kraj,Kopřivnice,Kopřivnice,2019-02-01T07:41:30.9430000+01:00,...,None,None,None,None,None,90820,None,2021-02-01,true,1
6,CZ-3848-19-02-0011,2019-02-01,Pravidelná,Plný,49369,3848,Moravskoslezský kraj,Kopřivnice,Kopřivnice,2019-02-01T08:22:14.2470000+01:00,...,None,None,None,None,None,85407,None,2021-02-01,true,1
7,CZ-3848-19-02-0001,2019-02-01,Pravidelná,Plný,49369,3848,Moravskoslezský kraj,Kopřivnice,Kopřivnice,2019-02-01T06:01:02.5500000+01:00,...,None,None,None,None,None,84630,None,2021-02-01,true,1
8,CZ-3848-19-02-0006,2019-02-01,Pravidelná,Plný,49369,3848,Moravskoslezský kraj,Kopřivnice,Kopřivnice,2019-02-01T07:21:48.2870000+01:00,...,None,None,None,None,None,86289,None,2021-02-01,true,1
9,CZ-3848-19-02-0002,2019-02-01,Pravidelná,Plný,49369,3848,Moravskoslezský kraj,Kopřivnice,Kopřivnice,2019-02-01T06:33:16.4270000+01:00,...,None,None,None,None,None,228053,None,2021-02-01,true,1


,0
CisloProtokolu,948272 / 948272
DatumProhlidky,948272 / 948272
DruhProhlidky,948272 / 948272
RozsahProhlidky,948272 / 948272
Prohlidka_OdpovednaOsoba,948272 / 948272
Prohlidka_Stanice_Cislo,948272 / 948272
Prohlidka_Stanice_Kraj,948272 / 948272
Prohlidka_Stanice_ORP,948272 / 948272
Prohlidka_Stanice_Obec,948272 / 948272
Prohlidka_Zahajeni,948272 / 948272


CisloProtokolu,DatumProhlidky,DruhProhlidky,RozsahProhlidky,Prohlidka_OdpovednaOsoba,Prohlidka_Stanice_Cislo,Prohlidka_Stanice_Kraj,Prohlidka_Stanice_ORP,Prohlidka_Stanice_Obec,Prohlidka_Zahajeni,Prohlidka_Ukonceni,AdministrativniOprava_CisloProtokolu,AdministrativniOprava_DatumProhlidky,Vozidlo_Vin,Vozidlo_Druh,Vozidlo_Kategorie,Vozidlo_Provedeni,Vozidlo_Znacka,Vozidlo_ObchodniOznaceni,Vozidlo_TypMotoru,Registrace_DatumPrvni,Registrace_Stat,Registrace_CisloDokladu,Emise_CisloProtokolu,Emise_DatumProhlidky,Emise_Stanice_Cislo,Emise_Zahajeni,Emise_Ukonceni,Emise_OdpovednaOsoba,Emise_ZakladniPalivo,Emise_AlternativniPalivo,Emise_EmisniSystem,Emise_VyrobceMotoru,Emise_CisloMotoru,Technicka_Zahajeni,Technicka_Ukonceni,Technicka_OdpovednaOsoba,Adr_Zahajeni,Adr_Ukonceni,Adr_OdpovednaOsoba,Adr_Platnost_Periodicka,Adr_Platnost_Meziperiodicka,Adr_KodCisterny,Adr_CisloOsvedceni,Adr_ZavadyText,Adr_Poznamka,Tsk_OdpovednaOsoba,Vysledek_Odometr,Vysledek_Poznamka,Vysledek_DatumPristiProhlidky,Vysledek_NalepkaVylepena,Vysledek_Celkovy
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""CZ-3848-19-02-0017""","""2019-02-01""","""Pravidelná""","""Plný""","""49369""","""3848""","""Moravskoslezský kraj""","""Kopřivnice""","""Kopřivnice""","""2019-02-01T10:12:21.8470000+01…","""2019-02-01T11:23:31.2300000+01…",null,null,"""VF7JMNFSC97165791""","""OSOBNÍ AUTOMOBIL""","""M1""",null,"""CITROËN""","""C2""","""NFS""","""2005-01-27T00:00:00.0000000+01…","""Česká republika""","""UG 217468""","""CZ-003848-19-02-0017""","""2019-02-01T11:23:31.2300000+01…","""3848""","""2019-02-01T10:14:14.4900000+01…","""2019-02-01T10:19:57.5630000+01…","""693""","""Benzín""",null,"""Řízený s OBD""",null,null,"""2019-02-01T10:51:34.0000000+01…","""2019-02-01T11:22:30.0000000+01…","""693""",null,null,null,null,null,null,null,null,null,null,"""223122""",null,"""2021-02-01""","""true""","""1"""


### Odstranění prázdných sloupců

In [5]:
adr_cols = [col for col in df.columns if 'Adr' in col]
df = df.drop(adr_cols + ['Tsk_OdpovednaOsoba'])

### Převedení sloupců týkajících se technické prohlídky na indikátor přítomnosti z důvodu řídkého výskytu

In [6]:
technicka_cols = ['RozsahProhlidky', 'Technicka_Zahajeni', 'Technicka_Ukonceni', 'Technicka_OdpovednaOsoba', 'Vysledek_NalepkaVylepena']
df = df.with_columns(pl.any_horizontal(pl.col(technicka_cols).is_not_null()).alias('Technicka_Pritomo')).drop(technicka_cols)
short_display(df)
display_counts(df)

(19748587, 38)


,CisloProtokolu,DatumProhlidky,DruhProhlidky,Prohlidka_OdpovednaOsoba,Prohlidka_Stanice_Cislo,Prohlidka_Stanice_Kraj,Prohlidka_Stanice_ORP,Prohlidka_Stanice_Obec,Prohlidka_Zahajeni,Prohlidka_Ukonceni,...,Emise_ZakladniPalivo,Emise_AlternativniPalivo,Emise_EmisniSystem,Emise_VyrobceMotoru,Emise_CisloMotoru,Vysledek_Odometr,Vysledek_Poznamka,Vysledek_DatumPristiProhlidky,Vysledek_Celkovy,Technicka_Pritomo
0,CZ-520406-19-01-0001,2019-01-01,Pravidelná,16835,520406,Středočeský kraj,Kolín,Žabonosy,2019-01-01T15:21:49.0770000+01:00,2019-01-01T15:40:05.4700000+01:00,...,Nafta,None,Řízený s OBD,None,None,309068,None,2021-01-01,1,False
1,CZ-420930-19-01-0001,2019-01-01,Pravidelná,43281,420930,Středočeský kraj,Říčany,Říčany,2019-01-01T14:35:04.4400000+01:00,2019-01-01T15:06:17.8430000+01:00,...,Nafta,None,Řízený bez OBD,None,None,169952,None,2021-01-01,1,False
2,CZ-480811-23-01-0001,2023-01-01,Pravidelná,36479,480811,Olomoucký kraj,Přerov,Kojetín,2023-01-01T12:21:49.8570000+01:00,2023-01-01T12:36:59.8530000+01:00,...,Nafta,None,Řízený s OBD,None,None,241379,None,2025-01-01,1,False
3,CZ-420221-23-01-0004,2023-01-01,Pravidelná,44365,420221,Středočeský kraj,Beroun,Bavoryně,2023-01-01T17:25:41.6570000+01:00,2023-01-01T17:42:24.7630000+01:00,...,Nafta,None,Řízený s OBD,None,-,65923,None,2025-01-01,1,False
4,CZ-420221-23-01-0001,2023-01-01,Pravidelná,44365,420221,Středočeský kraj,Beroun,Bavoryně,2023-01-01T16:05:05.4600000+01:00,2023-01-01T16:20:09.7700000+01:00,...,Nafta,None,Řízený s OBD,None,-,264005,None,2025-01-01,1,False
5,CZ-420221-23-01-0003,2023-01-01,Pravidelná,44365,420221,Středočeský kraj,Beroun,Bavoryně,2023-01-01T16:56:47.6530000+01:00,2023-01-01T17:07:00.6770000+01:00,...,Nafta,None,Řízený s OBD,None,-,146446,None,2025-01-01,1,False
6,CZ-170901-23-01-0001,2023-01-01,Pravidelná,89493,170901,Olomoucký kraj,Prostějov,Bedihošť,2023-01-01T19:23:52.2530000+01:00,2023-01-01T19:36:57.2500000+01:00,...,Nafta,None,Řízený s OBD,None,None,237886,None,2025-01-01,1,False
7,CZ-170901-23-01-0002,2023-01-01,Pravidelná,89493,170901,Olomoucký kraj,Prostějov,Bedihošť,2023-01-01T19:40:36.1970000+01:00,2023-01-01T19:53:31.8530000+01:00,...,Nafta,None,Řízený s OBD,None,None,210028,None,2025-01-01,1,False
8,CZ-520406-24-01-0001,2024-01-01,Pravidelná,1308,520406,Středočeský kraj,Kolín,Žabonosy,2024-01-01T11:30:09.4530000+01:00,2024-01-01T11:49:50.8200000+01:00,...,Benzín,None,Řízený s OBD,None,None,237115,None,2026-01-01,1,False
9,CZ-480811-24-01-0001,2024-01-01,Pravidelná,36479,480811,Olomoucký kraj,Přerov,Kojetín,2024-01-01T16:21:48.9530000+01:00,2024-01-01T16:36:21.9500000+01:00,...,Nafta,None,Řízený s OBD,None,None,263886,None,2026-01-01,1,False


,0
CisloProtokolu,19748587
DatumProhlidky,19748587
DruhProhlidky,19748587
Prohlidka_OdpovednaOsoba,19748587
Prohlidka_Stanice_Cislo,19748587
Prohlidka_Stanice_Kraj,19748587
Prohlidka_Stanice_ORP,19748587
Prohlidka_Stanice_Obec,19748587
Prohlidka_Zahajeni,19748587
Prohlidka_Ukonceni,19748587


# Podmínka pro osobní dieselová vozidla
## Analýza relevantních sloupců

In [7]:
df.filter(pl.col('Vozidlo_Druh') == 'OSOBNÍ AUTOMOBIL')['Vozidlo_Kategorie'].value_counts()

Vozidlo_Kategorie,count
str,u32
"""M1""",16044202
"""M1G""",483923
"""LA""",1
"""M2""",1
"""N1""",48
"""N3""",1


In [8]:
df['Emise_ZakladniPalivo'].value_counts()

Emise_ZakladniPalivo,count
str,u32
"""Směs""",36231
"""LNG""",399
"""CNG""",64512
"""Benzín""",9269023
"""Nafta""",10378161
"""LPG""",261


In [9]:
df['Emise_AlternativniPalivo'].value_counts()

Emise_AlternativniPalivo,count
str,u32
null,19299054
"""LPG""",416837
"""LNG""",144
"""CNG""",32552


## Filtrace na základě daných sloupců

In [10]:
df_diesel = df.filter((pl.col('Vozidlo_Druh') == 'OSOBNÍ AUTOMOBIL') & (pl.col('Emise_ZakladniPalivo') == 'Nafta') & pl.col('Emise_AlternativniPalivo').is_null())

### Selekce relevantního sloupce pro filtraci

In [13]:
short_display(df_diesel[['CisloProtokolu', 'Emise_CisloProtokolu']].filter(pl.col('CisloProtokolu') != pl.col('Emise_CisloProtokolu')))

(337376, 2)


,CisloProtokolu,Emise_CisloProtokolu
0,CZ-3848-19-02-0012,CZ-003848-19-02-0012
1,CZ-3848-19-02-0010,CZ-003848-19-02-0010
2,CZ-3848-19-02-0006,CZ-003848-19-02-0006
3,CZ-3848-19-02-0002,CZ-003848-19-02-0002
4,CZ-3618-21-02-0090,CZ-003618-21-02-0090
5,CZ-3618-21-02-0089,CZ-003618-21-02-0089
6,CZ-3706-21-02-0046,CZ-003706-21-02-0046
7,CZ-3644-21-02-0022,CZ-003644-21-02-0022
8,CZ-3706-21-02-0048,CZ-003706-21-02-0048
9,CZ-3840-21-02-0023,CZ-003840-21-02-0023


Pouze v jednom případě se čísla protokolu u emise a prohlídky liší. V datasetu dat z měření se vyskytuje Emise_CisloProtokolu. (Zajmavé je, že zrovna daný záznam je duplicitní, což indikuje, že by se mohlo jednat o chybu.)

In [30]:
short_display(df_diesel[['CisloProtokolu', 'Emise_CisloProtokolu']].filter(pl.col('CisloProtokolu') != pl.col('Emise_CisloProtokolu').str.replace(r'^CZ-(0+)(\d+)', r'CZ-${2}')))

(1, 2)


,CisloProtokolu,Emise_CisloProtokolu
0,CZ-3420-24-02-0512,CZ-540308-24-02-0512


Téměř všechny záznamy z datasetu dat z měřících přístrojů mají ekvivalentní záznam v datasetu prohlídek - dává tedy smysl použít sloupce z datasetu prohlídek pro filtrace osobních dieselových vozidel.

In [ ]:
# cislo_protokolu_technicka = set(df_technicka.select(pl.col('Emise_CisloProtokolu').str.replace(r'^CZ-(0+)(\d+)', r'CZ-${2}'))['Emise_CisloProtokolu'])
# cislo_protokolu_mereni = set(df_all.select(pl.col('CisloProtokolu').str.replace(r'^CZ-(0+)(\d+)', r'CZ-${2}'))['CisloProtokolu'])
# cislo_protokolu_mereni - cislo_protokolu_technicka

# RESULT:
# {'CZ-120902-20-07-0174', 'CZ-410432-20-08-0152'}